# Vestibular-BR — Treino SFT + GRPO com Unsloth (Kaggle)

**Configuração:** Accelerator = **GPU T4 x2** (o treino usa 1 GPU), Internet = **On**, Secret `HF_TOKEN` (permissão *write*).

Ordem dos experimentos (cada um vira um modelo no HF Hub):
| Experimento | Começa de | Treino |
|---|---|---|
| `sft` | Qwen3.5-2B | imitação dos raciocínios corretos do professor |
| `grpo` | Qwen3.5-2B | só aprendizado por reforço |
| `sft-grpo` | resultado do SFT | SFT e depois GRPO (receita estilo DeepSeek-R1) |

⏱️ O GRPO é a parte lenta: comece com `--max-steps 30` para conferir que roda, depois aumente.
Se o tempo da sessão (12h) acabar, os checkpoints ficam em `outputs/*/checkpoint-*`.

In [ ]:
# >>> TROQUE pelo seu usuário do GitHub e do Hugging Face <<<
GITHUB_REPO = "https://github.com/SEU_USUARIO/vestibular-br.git"
HF_USER = "SEU_USUARIO_HF"

import os
from kaggle_secrets import UserSecretsClient  # Add-ons > Secrets: crie HF_TOKEN (e GEMINI_API_KEY se quiser)
secrets = UserSecretsClient()
for k in ["HF_TOKEN", "GEMINI_API_KEY"]:
    try:
        os.environ[k] = secrets.get_secret(k)
    except Exception:
        print(f"secret {k} não configurado (ok se não for usar)")

In [ ]:
!git clone -q $GITHUB_REPO repo
%cd repo
!pip install -q unsloth
!pip install -q -e .
!vbr data

## Dataset de SFT
Gerado antes com `vbr distill` (roda na sua máquina, só usa API) e commitado em `data/sft/`.
Se ainda não existir, rode aqui mesmo (precisa do secret GEMINI_API_KEY):

In [ ]:
import os
if not os.path.exists("data/sft/train_sft.jsonl"):
    !vbr distill -m gemini:gemini-2.5-flash --k 4
!wc -l data/sft/train_sft.jsonl

## Experimento 1 — SFT

In [ ]:
!python -m vestibular.train.sft --out outputs/sft --push-to {HF_USER}/vestibular-br-2b-sft

## Experimento 2 — GRPO a partir do modelo base
`--difficulty-from` usa as predições com 4 amostras do notebook de avaliação (copie `results/runs/` para o repo antes).
Se não tiver, remova o argumento (treina em tudo, só é menos eficiente).

In [ ]:
import glob
diff = glob.glob("results/runs/train__vllm-Qwen-Qwen3.5-2B__caption__pass4/predictions.jsonl")
diff_arg = f"--difficulty-from {diff[0]}" if diff else ""
!python -m vestibular.train.grpo --out outputs/grpo --max-steps 250 {diff_arg} --push-to {HF_USER}/vestibular-br-2b-grpo

## Experimento 3 — SFT + GRPO

In [ ]:
!python -m vestibular.train.grpo --init-adapter outputs/sft --out outputs/sft-grpo --max-steps 250 {diff_arg} --push-to {HF_USER}/vestibular-br-2b-sft-grpo

In [ ]:
# Guarda as curvas de treino para o relatório
!mkdir -p results/training/grpo results/training/sft-grpo
!cp outputs/grpo/log_history.json results/training/grpo/ ; cp outputs/sft-grpo/log_history.json results/training/sft-grpo/
!zip -qr /kaggle/working/treino.zip results/training